<a href="https://colab.research.google.com/github/Islamintech/AI/blob/main/AI_Assignment2_MLP_20252403_%EC%9D%B4%EC%8A%AC%EB%A1%AC%EB%B2%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q datasets

import torch
from datasets import load_dataset

hf_train = load_dataset('uoft-cs/cifar10', split='train')
hf_test = load_dataset('uoft-cs/cifar10', split='test')

class HFCifar10(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.ds = hf_dataset
        self.transform = transform
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item['img'].convert('RGB')
        label = item['label']
        if self.transform:
            img = self.transform(img)
        return img, label

README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.9MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets


transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_datset = HFCifar10(hf_train, transform=transform_train)
test_dataset = HFCifar10(hf_test, transform=transform_test)


class MLP(nn.Module):
  def __init__(self):
    super(MLP, self).__init__()
    torch.manual_seed(2026)
    self.fc1 = nn.Linear(32 * 32 * 3, 1024)
    self.bn1 = nn.BatchNorm1d(1024)
    self.fc2 = nn.Linear(1024, 512)
    self.bn2 = nn.BatchNorm1d(512)
    self.fc3 = nn.Linear(512, 256)
    self.bn3 = nn.BatchNorm1d(256)
    self.fc4 = nn.Linear(256, 10)
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(0.3)

  def forward(self, x):
    x = x.view(-1, 32*32*3)
    x = self.relu(self.bn1(self.fc1(x)))
    x = self.dropout(x)
    x = self.relu(self.bn2(self.fc2(x)))
    x = self.dropout(x)
    x = self.relu(self.bn3(self.fc3(x)))
    return x

model = MLP().cuda()


train_batchsize = 128
train_loader = torch.utils.data.DataLoader(dataset=train_datset, batch_size=train_batchsize, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 30

model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pred = outputs.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    print(f'Epoch [{epoch+1}/{num_epochs}]  Loss: {running_loss/len(train_loader):.4f}  Train Acc: {100*correct/total:.2f}%')


batchsize = 64
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                           batch_size=batchsize, shuffle=True)

num_correct = 0

model.eval()

with torch.no_grad():
  for image, label in test_loader:
    image = image.cuda()
    label = label.cuda()
    output = model(image)
    pred = output.argmax(dim=1)
    num_correct += (pred == label).sum()

print(f'Accuracy : {num_correct / len(test_dataset) * 100:.2f} %')


Epoch [1/30]  Loss: 4.1277  Train Acc: 38.32%
Epoch [2/30]  Loss: 3.1336  Train Acc: 46.45%
Epoch [3/30]  Loss: 2.5967  Train Acc: 48.64%
Epoch [4/30]  Loss: 2.2248  Train Acc: 50.57%
Epoch [5/30]  Loss: 1.9703  Train Acc: 52.06%
Epoch [6/30]  Loss: 1.7876  Train Acc: 53.20%
Epoch [7/30]  Loss: 1.6705  Train Acc: 53.70%
Epoch [8/30]  Loss: 1.5733  Train Acc: 54.62%
Epoch [9/30]  Loss: 1.5045  Train Acc: 55.07%
Epoch [10/30]  Loss: 1.4526  Train Acc: 55.72%
Epoch [11/30]  Loss: 1.4074  Train Acc: 55.88%
Epoch [12/30]  Loss: 1.3674  Train Acc: 56.41%
Epoch [13/30]  Loss: 1.3441  Train Acc: 56.55%
Epoch [14/30]  Loss: 1.3175  Train Acc: 57.11%
Epoch [15/30]  Loss: 1.2990  Train Acc: 57.17%
Epoch [16/30]  Loss: 1.2823  Train Acc: 57.46%
Epoch [17/30]  Loss: 1.2587  Train Acc: 57.92%
Epoch [18/30]  Loss: 1.2438  Train Acc: 58.21%
Epoch [19/30]  Loss: 1.2367  Train Acc: 58.33%
Epoch [20/30]  Loss: 1.2259  Train Acc: 58.34%
Epoch [21/30]  Loss: 1.2080  Train Acc: 58.73%
Epoch [22/30]  Loss: 1